# Automated News Report Generator



## Problem

This notebook is the first stage of an automated pipeline that scrapes, summarizes and publishes a daily PDF report of the most relevant Spanish news. It's a portfolio project built to practice web scraping, data engineering and orchestration and MLOps tooling (Prefect, Docker, GitHub Actions, Hugging Face Spaces).

## Data source and credit

All news content used in this notebook is sourced from **RTVE (Radiotelevisión Española)**, the Spanish public broadcaster, through the public news sitemap published at rtve.es. All rights over the original articles belong to RTVE. This notebook stores the full article text only for internal use; the full text is never redistributed publicly. Every news item in the final report links back to the original RTVE article as the source of record.

**For verified and complete information, always refer to RTVE directly** (rtve.es). This project produces short, automated summaries; they are not a substitute for the original reporting.

## What this notebook does

1. Downloads RTVE's Google News sitemap with a self-identifying User-Agent (not the classic RSS feed, see the design decision below).
2. Parses the XML and keeps only news published in the last 48 hours.
3. Extracts id, section, title and publication date for each article.
4. Stores everything in a local SQLite database (`news.db`), designed to be idempotent (safe to re-run without duplicating rows).
5. Scrapes the full body and short description of each article with Playwright and updates the database.

## Key design decision (verified, not assumed)

The original plan was to use RTVE's classic RSS feed via `feedparser`. Checking RTVE's actual `robots.txt` showed that the classic feed redirects to `api2.rtve.es`, a host that is both stale (last updated in 2022) and blocked by its own `robots.txt`. Instead, this notebook uses RTVE's **Google News sitemap** (`https://www.rtve.es/sitemaps/sitemaps-news.xml`), which is explicitly allowed (`Allow: /sitemaps/*.xml$`), live, and links directly to each article page.

## Legal note

`robots.txt` only grants technical crawling permission, it says nothing about usage rights over the content. RTVE's legal notice prohibits reproducing its content without authorization. That's why the full article body (`Body`) is stored only for internal use and is never published as-is in the final PDF or dashboard. What gets published downstream is only the title, an AI-generated summary, the source (RTVE), and a link to the original article.


In [ ]:
# Data analysis libraries (inherited from the project template)
import pandas as pd
import numpy as np
import random

# HTTP download of the sitemap
import requests

# Parsing the sitemap XML (sitemap.org + Google News namespaces)
import xml.etree.ElementTree as ET

# Dates: recency filtering and pipeline timestamps
from datetime import timezone, datetime, timedelta

# Persistence for the extracted news
import sqlite3

# Scraping the article body. Using the ASYNC API (async_api) because
# Jupyter already runs its own asyncio event loop, and Playwright's
# sync API (sync_playwright) is incompatible with that inside a notebook
# (it works fine in a plain terminal script, just not here).
import playwright
from playwright.async_api import async_playwright


In [ ]:

SEED = 123
np.random.seed(SEED)
random.seed(SEED)

## 1. Downloading the news sitemap

The sitemap is downloaded with `requests`, sending a **self-identifying User-Agent** (`AutomatedNewsReportBot/1.0`) instead of Python's default one.

This isn't cosmetic: checking RTVE's `robots.txt` showed it explicitly blocks `User-Agent: Python-urllib` (the default used by `feedparser`/`urllib`) with `Disallow: /` for the entire site. Identifying with a proper User-Agent is also good scraping etiquette: anyone reviewing RTVE's server logs can tell what this bot is and how to reach out.


In [ ]:
# GET request with a custom User-Agent (see markdown cell above for why).
# 200 = OK, request went through without being blocked.
x = requests.get('https://www.rtve.es/sitemaps/sitemaps-news.xml', headers={"User-Agent":"AutomatedNewsReportBot/1.0 (https://github.com/AlejandroBeldaFernandez/Automated-News-Report)"})
print(x.status_code)

In [ ]:
# Peek at the raw XML to locate the namespace declarations (xmlns) before
# trying to query the tree with ElementTree.
print(x.text[:500])

In [ ]:
# XML namespace map. ElementTree needs this to resolve tags like
# "sitemaps:url" or "labels:title" to their full {uri}tag form internally.
# "sitemaps" -> the standard sitemap.org namespace (url, loc)
# "labels"   -> the Google News extension namespace (news, title, publication_date)
dicc = {"sitemaps": "http://www.sitemaps.org/schemas/sitemap/0.9", "labels": "http://www.google.com/schemas/sitemap-news/0.9"}

In [ ]:
# Parse the XML text into a navigable tree. `root` is the <urlset> element.
# Note the tags print with the namespace URI prefixed in {curly braces},
# which is exactly why the namespace map above is needed for any find/findall.
root = ET.fromstring(x.text)
print(root.tag)
print(root[0].tag)

In [ ]:
# All <url> entries in the sitemap: one per news item.
urls = root.findall('sitemaps:url', dicc)


### Trying field extraction on a single entry

Before looping over all entries, extract `loc`, `news:title` and `news:publication_date` from the first item to confirm the namespace paths are correct.


In [ ]:
# The article URL for the first entry.
urls[0].find("sitemaps:loc", dicc).text


In [ ]:
# "news:title" is nested inside "news:news", so the path has to go through
# that intermediate element first.
urls[0].find("labels:news/labels:title", dicc).text

In [ ]:
# ISO 8601 timestamp with timezone offset, needed later for the recency filter.
urls[0].find("labels:news/labels:publication_date", dicc).text


## 2. Building the news list: parsing, recency filter and field extraction

For every entry in the sitemap:
- Parse `loc`, `news:title` and `news:publication_date`.
- **Recency filter**: RTVE's sitemap is *not* limited to the last 48 hours as Google News sitemaps typically are (verified live: it mixed articles from 2008 with today's). So the 48h cutoff is enforced here in the pipeline, not assumed from the source.
- **`section`** and **`rtve_id`**: derived from the URL itself, since the sitemap doesn't provide them as separate fields. `rtve_id` extraction is defensive: if the last path segment isn't purely numeric, it's set to `None` and logged instead of crashing the whole loop, since this is external data RTVE could change.
- **`discovered_at`**: timestamp of this extraction, used later as the "phase 1" insertion time in SQLite.


In [ ]:
news = []
for item in urls:
    url = item.find("sitemaps:loc", dicc).text
    title = item.find("labels:news/labels:title", dicc).text
    published_at = item.find("labels:news/labels:publication_date", dicc).text
    publishet_at_correct = datetime.fromisoformat(published_at)
    actual_date = datetime.now(timezone.utc)

    # Recency filter: skip anything older than 48h (see markdown above,
    # the sitemap itself is not pre-filtered by RTVE).
    if (actual_date -  publishet_at_correct) > timedelta(hours=48):
        continue

    # section (e.g. "noticias", "deportes") is the 4th path segment.
    url_splitted = url.split("/")
    section = url_splitted[3]

    # rtve_id is the numeric id at the end of the URL, before ".shtml".
    # Defensive: don't crash on a URL that doesn't follow this pattern,
    # just flag it and store None.
    rtve_id = url_splitted[-1].split(".")[0]
    if rtve_id.isdigit():
        rtve_id = int(rtve_id)
    else:
        rtve_id = None
        print("Not Id in url: ", url)

    element = {"url": url, "title": title, "published_at": published_at, "section": section, "rtve_id": rtve_id, "discovered_at": actual_date.isoformat()}
    news.append(element)


## 3. Persisting to SQLite

Schema decisions, closed before writing any code:

- **`Url` is the `PRIMARY KEY`**, not `Rtve_id`. The URL (`loc`) is the only field the sitemap protocol always guarantees present and unique; the numeric id is something *we* derive from it and that extraction can fail, so it can't be a non-null primary key.
- **Two-phase schema**: `Body`, `Short_Description` and `Scrapped_at` are nullable because they only get filled later by Playwright (phase 2). Everything else is `NOT NULL` because it comes straight from the sitemap.
- **Idempotency**: `CREATE TABLE IF NOT EXISTS` + `INSERT OR IGNORE` so the whole notebook can be re-run (e.g. tomorrow, or after a retry in the future Prefect pipeline) without crashing or duplicating rows. This also doubles as the deduplication mechanism.


In [ ]:
# Connects to (or creates) a real database FILE, not an in-memory one,
# so the data persists across notebook runs.
conn = sqlite3.connect("news.db")
cur = conn.cursor()


In [ ]:
# IF NOT EXISTS: safe to re-run without erroring if the table is already there.
cur.execute("CREATE TABLE IF NOT EXISTS News (Url TEXT PRIMARY KEY NOT NULL, Rtve_id INTEGER , Title varchar(255) NOT NULL, Section varchar(255) NULL, Published_at varchar(255) NOT NULL, Discovered_at varchar(255) NOT NULL, Body varchar(255), Short_Description varchar(255), Scrapped_at varchar(255), Source varchar NOT NULL DEFAULT 'RTVE')")

In [ ]:
# Build one tuple per news item, values in the exact column order of the
# News table. Body/Short_Description/Scrapped_at are None here (phase 1),
# they get filled in by the Playwright step further down (phase 2).
tuples = []
for new in news:
    tuples.append((new['url'], new['rtve_id'], new['title'], new['section'], new['published_at'], new['discovered_at'], None, None, None, 'RTVE'))

In [ ]:
# OR IGNORE: if a row with this Url already exists (PRIMARY KEY clash),
# silently skip it instead of raising an IntegrityError. This is the
# deduplication mechanism decided for this pipeline.
cur.executemany("INSERT OR IGNORE INTO News VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)", tuples)

In [ ]:
# Commit once at the end of phase 1, not once per row.
conn.commit()

## 4. Scraping the article body with Playwright (single-article test)

Selectors verified by inspecting a real RTVE article page's HTML, not guessed:
- Article body: `.artBody` (a `<div class="artBody">` inside `.mainContent`).
- Short description: the `content` attribute of `<meta name="description">`.

Playwright's browser identifies itself as a normal Chrome browser by default, which is intentionally kept as-is (not overridden with the custom bot User-Agent used for the sitemap download). RTVE's `robots.txt` is served from an `/akamai/` path, a sign that a bot-management WAF sits in front of the site; a realistic browser User-Agent is less likely to be fingerprinted and blocked on pages meant for human visitors.

Tested here on a single article (`news[0]`) before running the full loop over every item.


In [ ]:
# NOTE: uses the ASYNC Playwright API (async with / await) because Jupyter
# already runs its own asyncio event loop; sync_playwright would raise
# "Sync API inside the asyncio loop" here (it's fine in a plain .py script).
async with async_playwright() as p:
    browser = await p.chromium.launch()
    page = await browser.new_page()
    await page.goto(news[0]["url"])
    body = await page.inner_text(".artBody")
    short_description = await page.get_attribute('meta[name="description"]', 'content')
    print(await page.title())
    await browser.close()

## 5. Full scraping loop and final persistence (phase 2)

Loops over every item in `news`, scrapes body + short description, and runs an `UPDATE` on the matching row (`WHERE Url = ?`) to fill in the phase-2 columns.

Design notes:
- The `browser` is launched **once** and reused for all articles (expensive to start), but a fresh `page` is opened and closed per article (cheap, avoids accumulating tabs).
- The `try/except` around the scraping call is the same defensive philosophy applied earlier to `rtve_id`: if one article's page fails to load or its HTML doesn't match the expected selectors, that single row just ends up with `Body`/`Short_Description` as `None`, the loop keeps going instead of crashing for the other ~75 articles.
- Known gap, not yet fixed: `.artBody` also contains "related articles" boxes (`.incluBox`) interleaved with the real paragraphs, so `Body` currently includes that noise. Left as an open item for a future text-cleaning step.
- `conn.commit()` runs once after the loop, not per row.


In [ ]:
async with async_playwright() as p:
    browser = await p.chromium.launch()

    for item in news:
        page = await browser.new_page()
        try:
            await page.goto(item["url"], timeout=30000)
            body = await page.inner_text(".artBody")
            short_description = await page.get_attribute('meta[name="description"]', "content")
        except Exception as e:
            # Defensive: one bad article must not stop the whole batch.
            print("Fail in", item["url"], ":", e)
            body = None
            short_description = None
        await page.close()

        scraped_at = datetime.now(timezone.utc).isoformat()
        cur.execute(
            "UPDATE News SET Body = ?, Short_Description = ?, Scrapped_at = ? WHERE Url = ?",
            (body, short_description, scraped_at, item["url"])
        )

    await browser.close()

conn.commit()


## Results

Quick sanity check on what actually landed in `news.db`, run against the real database rather than assumed.


In [63]:
# Row counts and how many rows completed phase 2 (Playwright scraping).
cur.execute("SELECT COUNT(*), COUNT(Body), COUNT(Short_Description) FROM News")
total, with_body, with_short_description = cur.fetchone()
print(f"Total news rows: {total}")
print(f"Rows with Body scraped: {with_body}")
print(f"Rows with Short_Description scraped: {with_short_description}")

cur.execute("SELECT Section, COUNT(*) FROM News GROUP BY Section ORDER BY COUNT(*) DESC")
print("\nBy section:")
for section, count in cur.fetchall():
    print(f"  {section}: {count}")

Total news rows: 77
Rows with Body scraped: 77
Rows with Short_Description scraped: 77

By section:
  noticias: 51
  play: 14
  deportes: 6
  catalunya: 4
  rtve: 2
